- LLM拓展——多提供商支持provider
    - 不同服务商在环境变量、api、模型等方面有差异，引入 provider，在内部处理不同服务商的配置细节
    - \* 见 my_llm.py 中，继承原AgentsLLM类，增加对modelscope平台的支持为例，实现在不修改源码的情况下，成功拓展了新功能。
- LLM拓展——本地模型调用
    - VLLM 和 Ollama 可以有效下载管理本地模型，并兼容OpenAI标准的API服务
    - \* 见 my_llm.py 中，添加了provider是ollama时连接本地服务
- LLM拓展——自动检测机制
    - 根据特定服务商的环境变量判断，依次检查 MODELSCOPE_API_KEY, OPENAI_API_KEY, ZHIPU_API_KEY等环境变量是否存在；
    - 根据base_url判断，域名/端口；
    - 辅助判断，分析api密钥格式

- 框架接口实现：
    - Message类：定义框架内统一的消息格式，确保信息传递标准化，规范管理上下文 (\* 见 message.py)
    - Config类：配置管理方案，使框架易于调整和拓展，将代码中硬编码配置参数集中起来，并支持从环境变量中读取 (\* 见config.py)
    - Agent基类：定义了所有agent基类，通过python的abc模块实现，为后续实现不同类型的agent提供统一接口和规范 (\* agent.py)

- agent范式框架化实现：三种经典范式(ReAct, Plan&Solve, Reflection)进行框架化重构，并新增SimpleAgent作为基础对话范式。在导入继承hello_agents库中的框架基类的基础上，学习在框架规范上扩展：
    - SimpleAgent：最基础，构建一个完整的对话智能体。通过继承框架中已有的 SimpleAgent 类并重写其核心方法，实现可拓展的版本
      (\* 见 my_simple_agent.py，另见 test_simple_agent.ipynb 测试)
    - ReActAgent：主要进行提示词优化和框架工具系统集成
      (\* 见 my_react_agent.py，另见 test_react_agent.ipynb 测试)
    - Reflection：
    - Plan-and-Solve：

- 工具系统：标准化的工具Tool基类与ToolRegistry注册机制
    - Tool基类的抽象设计：
    ```
    class Tool(ABC):
        """工具基类"""

        def __init__(self, name: str, description: str):
            self.name = name
            self.description = description

        @abstractmethod
        def run(self, parameters: Dict[str, Any]) -> str:
            """执行工具"""
            pass

        @abstractmethod
        def get_parameters(self) -> List[ToolParameter]:
            """获取工具参数定义"""
            pass
    ```
    - ToolParameter参数定义系统：
    ```
    class ToolParameter(BaseModel):
        """工具参数定义"""
        name: str
        type: str
        description: str
        required: bool = True
        default: Any = None
    ```
    - ToolRegistry注册表的实现：
    ```
    class ToolRegistry:
        """HelloAgents工具注册表"""
        def __init__(self):
            self._tools: dict[str, Tool] = {}
            self._functions: dict[str, dict[str, Any]] = {}

        def register_tool(self, tool: Tool):
            """注册Tool对象"""
            pass

        def register_function(self, name: str, description: str, func: Callable[[str], str]):
            """直接注册函数作为工具（简便方式）"""
            pass

        def get_tools_description(self) -> str:
            """获取所有可用工具的格式化描述字符串"""
            pass
    ```